# Preflop Ranges — 6-Max Tournaments

Visualization of opening ranges by position in 6-max.  
The 13×13 matrix represents the 169 hand types:
- **Diagonal**: pairs (AA, KK, …)
- **Upper triangle**: suited hands (AKs, A5s, …)
- **Lower triangle**: offsuit hands (AKo, A5o, …)

In [1]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Constants ─────────────────────────────────────────────────────────────────
RANKS = ['A', 'K', 'Q', 'J', 'T', '9', '8', '7', '6', '5', '4', '3', '2']
N = len(RANKS)  # 13

def hand_name(i, j):
    if i == j:
        return RANKS[i] * 2
    elif i < j:
        return RANKS[i] + RANKS[j] + 's'
    else:
        return RANKS[j] + RANKS[i] + 'o'

HAND_MATRIX = [[hand_name(i, j) for j in range(N)] for i in range(N)]

print("Top-left corner    :", HAND_MATRIX[0][0])    # AA
print("Top-right corner   :", HAND_MATRIX[0][12])   # A2s
print("Bottom-left corner :", HAND_MATRIX[12][0])   # A2o
print("Bottom-right corner:", HAND_MATRIX[12][12])  # 22

Top-left corner    : AA
Top-right corner   : A2s
Bottom-left corner : A2o
Bottom-right corner: 22


## LJ (Lojack) — 6-Max Opening Range

First to act in 6-max. Tight range, around **20-22%** of hands.

In [2]:
# ── LJ 6-Max Ranges ───────────────────────────────────────────────────────────
# GREEN : standard open
LJ_GREEN = {
    # Pairs 55+
    '55', '66', '77', '88', '99', 'TT', 'JJ', 'QQ', 'KK', 'AA',
    # Suited aces A2s+
    'A2s', 'A3s', 'A4s', 'A5s', 'A6s', 'A7s', 'A8s', 'A9s', 'ATs', 'AJs', 'AQs', 'AKs',
    # Offsuit broadways
    'AJo', 'AQo', 'AKo', 'KQo',
    # Suited kings KTs+
    'KTs', 'KJs', 'KQs',
    # Suited queens QTs+
    'QTs', 'QJs',
    # Suited connectors / gappers
    'JTs', 'J9s', 'T9s', '98s', '87s',
}

# YELLOW : open if table is loose / exploitable
LJ_YELLOW = {
    # Small pairs
    '22', '33', '44',
    # Offsuit
    'ATo', 'KJo',
    # Suited kings / queens
    'K9s', 'Q9s',
    # Suited connectors
    '76s', '65s', 'T8s', '97s',
}

# ── Combos & statistics ───────────────────────────────────────────────────────
def count_combos(hand_set):
    total = 0
    for h in hand_set:
        if h[0] == h[1]:    total += 6   # pair
        elif h.endswith('s'): total += 4  # suited
        else:               total += 12  # offsuit
    return total

combos_green  = count_combos(LJ_GREEN)
combos_yellow = count_combos(LJ_YELLOW)
combos_total  = combos_green + combos_yellow

print(f"Green  (open)        : {len(LJ_GREEN):>3} hands  {combos_green:>4} combos  {combos_green/1326*100:.1f} %")
print(f"Yellow (open/loose)  : {len(LJ_YELLOW):>3} hands  {combos_yellow:>4} combos  {combos_yellow/1326*100:.1f} %")
print(f"Total range          : {len(LJ_GREEN)+len(LJ_YELLOW):>3} hands  {combos_total:>4} combos  {combos_total/1326*100:.1f} %")

Green  (open)        :  36 hands   196 combos  14.8 %
Yellow (open/loose)  :  11 hands    66 combos  5.0 %
Total range          :  47 hands   262 combos  19.8 %


In [ ]:
def draw_range_matrix(green_set, yellow_set, title, filename=None):
    c_green  = count_combos(green_set)
    c_yellow = count_combos(yellow_set)
    c_open   = c_green + c_yellow

    fig, ax = plt.subplots(figsize=(18, 19))
    fig.patch.set_facecolor('#f8f9fa')
    ax.set_facecolor('#f8f9fa')
    fig.subplots_adjust(top=0.91, bottom=0.09, left=0.06, right=0.97)

    COLORS = {
        'green':  ('#27ae60', '#ffffff'),
        'yellow': ('#e67e22', '#ffffff'),
        'fold':   ('#dce1e7', '#8395a7'),
    }

    ax.set_xlim(-0.5, N)
    ax.set_ylim(-0.5, N)
    ax.set_aspect('equal')
    ax.axis('off')

    for i in range(N):
        for j in range(N):
            hand = HAND_MATRIX[i][j]
            fc, tc = (COLORS['green']  if hand in green_set  else
                      COLORS['yellow'] if hand in yellow_set else
                      COLORS['fold'])
            x, y = j, N - 1 - i
            ax.add_patch(mpatches.FancyBboxPatch(
                (x + 0.06, y + 0.06), 0.88, 0.88,
                boxstyle="round,pad=0.06",
                facecolor=fc, edgecolor='white', linewidth=2.5, zorder=2
            ))
            fs = 10.5 if len(hand) == 4 else 12.5
            ax.text(x + 0.5, y + 0.5, hand,
                    ha='center', va='center',
                    fontsize=fs, fontweight='bold', color=tc, zorder=3)

    # Rank labels
    for k, r in enumerate(RANKS):
        ax.text(k + 0.5, N + 0.1, r,
                ha='center', va='bottom', fontsize=14, fontweight='bold', color='#2c3e50')
        ax.text(-0.1, N - 1 - k + 0.5, r,
                ha='right', va='center', fontsize=14, fontweight='bold', color='#2c3e50')

    # Suited / offsuit diagonal separator
    for k in range(N):
        ax.plot([k, k + 1], [N - k, N - k],     color='#95a5a6', lw=1.2, zorder=4)
        ax.plot([k, k],     [N - k, N - k - 1], color='#95a5a6', lw=1.2, zorder=4)

    ax.text(N * 0.80, N - 0.22, 'suited ▲',
            fontsize=11, color='#636e72', ha='center', style='italic', zorder=5)
    ax.text(N * 0.20, -0.22, '▽ offsuit',
            fontsize=11, color='#636e72', ha='center', style='italic', zorder=5)

    # Title — above the matrix
    fig.text(0.5, 0.955, title,
             ha='center', va='top', fontsize=18, fontweight='bold', color='#2c3e50')
    fig.text(0.5, 0.935,
             f'{c_open} combos  ·  {c_open / 1326 * 100:.1f} % of hands',
             ha='center', va='top', fontsize=12, color='#636e72')

    # Legend — at the bottom, outside the matrix
    legend_items = [
        mpatches.Patch(facecolor='#27ae60', edgecolor='white',
                       label=f'Open  —  {c_green} combos  ({c_green / 1326 * 100:.1f} %)'),
        mpatches.Patch(facecolor='#e67e22', edgecolor='white',
                       label=f'Open if table is loose  —  {c_yellow} combos  ({c_yellow / 1326 * 100:.1f} %)'),
        mpatches.Patch(facecolor='#dce1e7', edgecolor='#b2bec3', label='Fold'),
    ]
    fig.legend(handles=legend_items, loc='lower center',
               bbox_to_anchor=(0.5, 0.01), ncol=3,
               fontsize=11, framealpha=0.97, edgecolor='#b2bec3',
               fancybox=True, handlelength=1.6, handleheight=1.2)

    fname = filename or title.split()[0].lower() + '_range.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()


draw_range_matrix(LJ_GREEN, LJ_YELLOW, 'LJ Open Raise — 6-Max Tournaments')

## HJ (Hijack) — 6-Max Opening Range

Second to act in 6-max. Slightly wider range than LJ, around **23-26%** of hands.

In [ ]:
# ── HJ 6-Max Ranges ───────────────────────────────────────────────────────────
HJ_GREEN = {
    # Pairs 44+
    '44', '55', '66', '77', '88', '99', 'TT', 'JJ', 'QQ', 'KK', 'AA',
    # Suited aces A2s+
    'A2s', 'A3s', 'A4s', 'A5s', 'A6s', 'A7s', 'A8s', 'A9s', 'ATs', 'AJs', 'AQs', 'AKs',
    # Offsuit aces ATo+
    'ATo', 'AJo', 'AQo', 'AKo',
    # Suited kings K8s+
    'K8s', 'K9s', 'KTs', 'KJs', 'KQs',
    # Offsuit
    'KQo',
    # Suited queens Q9s+
    'Q9s', 'QTs', 'QJs',
    # Suited jacks J9s+
    'J9s', 'JTs',
    # Suited connectors 87s+
    '87s', '98s', 'T9s',
    # One-gapper
    'T8s',
}

HJ_YELLOW = {
    # Small pairs
    '22', '33',
    # Suited kings
    'K6s', 'K7s',
    # Offsuit
    'KJo',
    # Suited queens / jacks
    'Q8s', 'J8s',
    # Suited connectors / gappers
    '65s', '76s', '86s', '97s',
}

combos_green  = count_combos(HJ_GREEN)
combos_yellow = count_combos(HJ_YELLOW)

print(f"Green  (open)        : {len(HJ_GREEN):>3} hands  {combos_green:>4} combos  {combos_green/1326*100:.1f} %")
print(f"Yellow (open/loose)  : {len(HJ_YELLOW):>3} hands  {combos_yellow:>4} combos  {combos_yellow/1326*100:.1f} %")
print(f"Total range          : {len(HJ_GREEN)+len(HJ_YELLOW):>3} hands  {combos_green+combos_yellow:>4} combos  {(combos_green+combos_yellow)/1326*100:.1f} %")

draw_range_matrix(HJ_GREEN, HJ_YELLOW, 'HJ Open Raise — 6-Max Tournaments')

## CO (Cutoff) — 6-Max Opening Range

Third position in 6-max. Wide range, around **28-32%** of hands.

In [ ]:
# ── CO 6-Max Ranges ───────────────────────────────────────────────────────────
CO_GREEN = {
    # All pairs 22+
    '22', '33', '44', '55', '66', '77', '88', '99', 'TT', 'JJ', 'QQ', 'KK', 'AA',
    # Suited aces A2s+
    'A2s', 'A3s', 'A4s', 'A5s', 'A6s', 'A7s', 'A8s', 'A9s', 'ATs', 'AJs', 'AQs', 'AKs',
    # Suited kings K5s+
    'K5s', 'K6s', 'K7s', 'K8s', 'K9s', 'KTs', 'KJs', 'KQs',
    # Suited queens Q8s+
    'Q8s', 'Q9s', 'QTs', 'QJs',
    # Suited jacks J8s+
    'J8s', 'J9s', 'JTs',
    # Suited tens T8s+
    'T8s', 'T9s',
    # 97s+
    '97s', '98s',
    # 86s+
    '86s', '87s',
    # 65s+
    '65s', '76s',
    # Offsuit aces A9o+
    'A9o', 'ATo', 'AJo', 'AQo', 'AKo',
    # Offsuit kings KTo+
    'KTo', 'KJo', 'KQo',
    # Offsuit queens
    'QJo',
}

CO_YELLOW = {
    # Offsuit aces
    'A7o', 'A8o',
    # Suited kings
    'K3s', 'K4s',
    # Offsuit king
    'K9o',
    # Suited queens / jacks
    'Q7s', 'J7s',
    # Offsuit
    'QTo', 'JTo',
    # Suited
    '54s', '75s',
}

combos_green  = count_combos(CO_GREEN)
combos_yellow = count_combos(CO_YELLOW)

print(f"Green  (open)        : {len(CO_GREEN):>3} hands  {combos_green:>4} combos  {combos_green/1326*100:.1f} %")
print(f"Yellow (open/loose)  : {len(CO_YELLOW):>3} hands  {combos_yellow:>4} combos  {combos_yellow/1326*100:.1f} %")
print(f"Total range          : {len(CO_GREEN)+len(CO_YELLOW):>3} hands  {combos_green+combos_yellow:>4} combos  {(combos_green+combos_yellow)/1326*100:.1f} %")

draw_range_matrix(CO_GREEN, CO_YELLOW, 'CO Open Raise — 6-Max Tournaments')

## BTN (Button) — 6-Max Opening Range

Best position. Very wide range, around **45-50%** of hands.

In [ ]:
# ── BTN 6-Max Ranges ──────────────────────────────────────────────────────────
BTN_GREEN = {
    # All pairs 22+
    '22', '33', '44', '55', '66', '77', '88', '99', 'TT', 'JJ', 'QQ', 'KK', 'AA',
    # Suited aces A2s+
    'A2s', 'A3s', 'A4s', 'A5s', 'A6s', 'A7s', 'A8s', 'A9s', 'ATs', 'AJs', 'AQs', 'AKs',
    # Suited kings K2s+
    'K2s', 'K3s', 'K4s', 'K5s', 'K6s', 'K7s', 'K8s', 'K9s', 'KTs', 'KJs', 'KQs',
    # Suited queens Q2s+
    'Q2s', 'Q3s', 'Q4s', 'Q5s', 'Q6s', 'Q7s', 'Q8s', 'Q9s', 'QTs', 'QJs',
    # Suited jacks J4s+
    'J4s', 'J5s', 'J6s', 'J7s', 'J8s', 'J9s', 'JTs',
    # Suited tens T5s+
    'T5s', 'T6s', 'T7s', 'T8s', 'T9s',
    # 95s+
    '95s', '96s', '97s', '98s',
    # 85s+
    '85s', '86s', '87s',
    # 74s+
    '74s', '75s', '76s',
    # 64s+
    '64s', '65s',
    # Low suited connectors
    '54s', '43s',
    # Offsuit aces A2o+
    'A2o', 'A3o', 'A4o', 'A5o', 'A6o', 'A7o', 'A8o', 'A9o', 'ATo', 'AJo', 'AQo', 'AKo',
    # Offsuit kings K7o+
    'K7o', 'K8o', 'K9o', 'KTo', 'KJo', 'KQo',
    # Offsuit queens Q9o+
    'Q9o', 'QTo', 'QJo',
    # Offsuit jacks J9o+
    'J9o', 'JTo',
    # Offsuit
    'T9o',
}

BTN_YELLOW = {
    # Offsuit kings
    'K5o', 'K6o',
    # Offsuit queens / jacks
    'Q8o', 'J8o',
    # Low suited jacks
    'J2s', 'J3s',
    # Low suited tens
    'T2s', 'T3s', 'T4s',
    # Low suited
    '32s',
}

combos_green  = count_combos(BTN_GREEN)
combos_yellow = count_combos(BTN_YELLOW)

print(f"Green  (open)        : {len(BTN_GREEN):>3} hands  {combos_green:>4} combos  {combos_green/1326*100:.1f} %")
print(f"Yellow (open/loose)  : {len(BTN_YELLOW):>3} hands  {combos_yellow:>4} combos  {combos_yellow/1326*100:.1f} %")
print(f"Total range          : {len(BTN_GREEN)+len(BTN_YELLOW):>3} hands  {combos_green+combos_yellow:>4} combos  {(combos_green+combos_yellow)/1326*100:.1f} %")

draw_range_matrix(BTN_GREEN, BTN_YELLOW, 'BTN Open Raise — 6-Max Tournaments')

## SB (Small Blind) — 6-Max Range

Most complex position: we play **3 actions** instead of 2.
- **Dark green**: Value raise (99+, ATs+, AJo+, KQs, KQo)
- **Light green**: Bluff raise (semi-bluff with offsuit blockers)
- **Blue**: Limp (unfavorable OOP, don't enter too expensively)

In [ ]:
# ── SB 6-Max Ranges — Optimized Version ──────────────────────────────────────
#
# Principles:
#   VALUE RAISE   → hands that want a 3-bet pot (strong HU equity vs wide BB)
#   BLUFF RAISE   → offsuit blockers only: we fold without regret to a 3-bet
#   LIMP          → hands with post-flop potential that don't need a big pot
#                   Suited connectors stay here — they LOVE cheap flops
#                   and lose too much value when raised OOP

# DARK GREEN: value raise (17 hands)
SB_VALUE = {
    # Pairs 88+ (88 raises: monster equity vs wide BB range)
    '88', '99', 'TT', 'JJ', 'QQ', 'KK', 'AA',
    # Suited: ATs+ and KQs
    'ATs', 'AJs', 'AQs', 'AKs', 'KQs',
    # Offsuit: AJo+ and KJo+
    'AJo', 'AQo', 'AKo', 'KJo', 'KQo',
}

# LIGHT GREEN: bluff raise (10 hands — offsuit blockers only)
SB_BLUFF = {
    # Offsuit ace blockers (A2-A5: clean fold to a 3-bet, blocks AA/AK/AQ)
    'A2o', 'A3o', 'A4o', 'A5o',
    # Offsuit king blockers K7-K9
    'K7o', 'K8o', 'K9o',
    # Offsuit connectors (semi-bluff, fold to a 3-bet)
    'QTo', 'JTo', 'T9o',
}

# BLUE: limp (70 hands)
SB_LIMP = {
    # Pairs 22-77 (88 moves up to value raise)
    '22', '33', '44', '55', '66', '77',
    # Suited aces A2s-A9s
    'A2s', 'A3s', 'A4s', 'A5s', 'A6s', 'A7s', 'A8s', 'A9s',
    # Suited kings K2s-KJs
    'K2s', 'K3s', 'K4s', 'K5s', 'K6s', 'K7s', 'K8s', 'K9s', 'KTs', 'KJs',
    # Suited queens Q2s-QJs
    'Q2s', 'Q3s', 'Q4s', 'Q5s', 'Q6s', 'Q7s', 'Q8s', 'Q9s', 'QTs', 'QJs',
    # Suited jacks J4s-J9s
    'J4s', 'J5s', 'J6s', 'J7s', 'J8s', 'J9s',
    # Premium suited connectors (limp: cheap flops > bluff raise OOP)
    'JTs', 'T9s', '98s', '87s',
    # Mid suited gappers
    'T6s', 'T7s', 'T8s',
    '96s', '97s',
    '85s', '86s',
    '75s', '76s',
    '64s', '65s',
    '53s', '54s',
    '43s', '32s',
    # Offsuit aces A6o-ATo
    'A6o', 'A7o', 'A8o', 'A9o', 'ATo',
    # KTo: limp (too thin to raise for value OOP, prefers a cheap flop)
    'KTo',
    # Offsuit kings defense
    'K5o', 'K6o',
    # Other offsuit
    'Q9o', 'QJo', 'J9o',
}

# ⚪ FOLD: everything else (87o, weak disconnected hands, etc.)

def draw_sb_matrix(value_set, bluff_set, limp_set, title, filename='sb_range.png'):
    cc = count_combos
    c_val, c_bluff, c_limp = cc(value_set), cc(bluff_set), cc(limp_set)

    fig, ax = plt.subplots(figsize=(18, 19))
    fig.patch.set_facecolor('#f8f9fa')
    ax.set_facecolor('#f8f9fa')
    fig.subplots_adjust(top=0.91, bottom=0.09, left=0.06, right=0.97)

    def cell_color(hand):
        if hand in value_set: return '#145a32', '#ffffff'
        if hand in bluff_set: return '#82e0aa', '#145a32'
        if hand in limp_set:  return '#2471a3', '#ffffff'
        return '#dce1e7', '#8395a7'

    ax.set_xlim(-0.5, N)
    ax.set_ylim(-0.5, N)
    ax.set_aspect('equal')
    ax.axis('off')

    for i in range(N):
        for j in range(N):
            hand = HAND_MATRIX[i][j]
            fc, tc = cell_color(hand)
            x, y = j, N - 1 - i
            ax.add_patch(mpatches.FancyBboxPatch(
                (x + 0.06, y + 0.06), 0.88, 0.88,
                boxstyle="round,pad=0.06",
                facecolor=fc, edgecolor='white', linewidth=2.5, zorder=2
            ))
            fs = 10.5 if len(hand) == 4 else 12.5
            ax.text(x + 0.5, y + 0.5, hand,
                    ha='center', va='center',
                    fontsize=fs, fontweight='bold', color=tc, zorder=3)

    for k, r in enumerate(RANKS):
        ax.text(k + 0.5, N + 0.1, r,
                ha='center', va='bottom', fontsize=14, fontweight='bold', color='#2c3e50')
        ax.text(-0.1, N - 1 - k + 0.5, r,
                ha='right', va='center', fontsize=14, fontweight='bold', color='#2c3e50')

    for k in range(N):
        ax.plot([k, k + 1], [N - k, N - k],     color='#95a5a6', lw=1.2, zorder=4)
        ax.plot([k, k],     [N - k, N - k - 1], color='#95a5a6', lw=1.2, zorder=4)

    ax.text(N * 0.80, N - 0.22, 'suited ▲',
            fontsize=11, color='#636e72', ha='center', style='italic', zorder=5)
    ax.text(N * 0.20, -0.22, '▽ offsuit',
            fontsize=11, color='#636e72', ha='center', style='italic', zorder=5)

    fig.text(0.5, 0.955, title,
             ha='center', va='top', fontsize=18, fontweight='bold', color='#2c3e50')
    fig.text(0.5, 0.935,
             f'Value raise {c_val} ({c_val/1326*100:.1f}%)  ·  '
             f'Bluff raise {c_bluff} ({c_bluff/1326*100:.1f}%)  ·  '
             f'Limp {c_limp} ({c_limp/1326*100:.1f}%)',
             ha='center', va='top', fontsize=11, color='#636e72')

    legend_items = [
        mpatches.Patch(facecolor='#145a32', edgecolor='white',
                       label=f'Value raise  —  {c_val} combos  ({c_val/1326*100:.1f} %)'),
        mpatches.Patch(facecolor='#82e0aa', edgecolor='#145a32',
                       label=f'Bluff raise  —  {c_bluff} combos  ({c_bluff/1326*100:.1f} %)'),
        mpatches.Patch(facecolor='#2471a3', edgecolor='white',
                       label=f'Limp  —  {c_limp} combos  ({c_limp/1326*100:.1f} %)'),
        mpatches.Patch(facecolor='#dce1e7', edgecolor='#b2bec3', label='Fold'),
    ]
    fig.legend(handles=legend_items, loc='lower center',
               bbox_to_anchor=(0.5, 0.01), ncol=4,
               fontsize=10.5, framealpha=0.97, edgecolor='#b2bec3',
               fancybox=True, handlelength=1.6, handleheight=1.2)

    plt.savefig(filename, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()


c_val   = count_combos(SB_VALUE)
c_bluff = count_combos(SB_BLUFF)
c_limp  = count_combos(SB_LIMP)
c_fold  = 1326 - c_val - c_bluff - c_limp
print(f"Value raise   : {len(SB_VALUE):>2} hands  {c_val:>4} combos  {c_val/1326*100:.1f} %")
print(f"Bluff raise   : {len(SB_BLUFF):>2} hands  {c_bluff:>4} combos  {c_bluff/1326*100:.1f} %")
print(f"Limp          : {len(SB_LIMP):>2} hands  {c_limp:>4} combos  {c_limp/1326*100:.1f} %")
print(f"Fold          :     —      {c_fold:>4} combos  {c_fold/1326*100:.1f} %")
print(f"────────────────────────────────────────────────────")
print(f"Total active  :     —      {c_val+c_bluff+c_limp:>4} combos  {(c_val+c_bluff+c_limp)/1326*100:.1f} %")

draw_sb_matrix(SB_VALUE, SB_BLUFF, SB_LIMP, 'SB — 6-Max Tournaments Range (Optimized)')

---
## BTN — IP Defense (facing an open)

The BTN is **In Position** post-flop against all openers (LJ, HJ, CO).  
Polarized strategy: we 3-bet strong hands or bluffs, call the rest, fold the trash.

- 🔴 **Value 3-bet**: hands that want a big pot (ahead of the opener's range)
- 🟠 **Bluff 3-bet**: blockers (offsuit Aces/Kings) — clean fold to a 4-bet
- 🟢 **Call**: hands with IP potential, speculative hands, medium pairs
- ⚪ **Fold**: everything else

> The later the opener's position (wider range), the wider the BTN 3-bets.

In [ ]:
def draw_defense_matrix(vbet_set, bluff_set, call_set, title, filename=None):
    cc = count_combos
    c_vbet  = cc(vbet_set)
    c_bluff = cc(bluff_set)
    c_call  = cc(call_set)
    c_fold  = 1326 - c_vbet - c_bluff - c_call

    fig, ax = plt.subplots(figsize=(18, 19))
    fig.patch.set_facecolor('#f8f9fa')
    ax.set_facecolor('#f8f9fa')
    fig.subplots_adjust(top=0.91, bottom=0.09, left=0.06, right=0.97)

    def cell_color(hand):
        if hand in vbet_set:  return '#922b21', '#ffffff'   # dark red
        if hand in bluff_set: return '#e67e22', '#ffffff'   # orange
        if hand in call_set:  return '#1e8449', '#ffffff'   # green
        return '#dce1e7', '#8395a7'                         # grey fold

    ax.set_xlim(-0.5, N)
    ax.set_ylim(-0.5, N)
    ax.set_aspect('equal')
    ax.axis('off')

    for i in range(N):
        for j in range(N):
            hand = HAND_MATRIX[i][j]
            fc, tc = cell_color(hand)
            x, y = j, N - 1 - i
            ax.add_patch(mpatches.FancyBboxPatch(
                (x + 0.06, y + 0.06), 0.88, 0.88,
                boxstyle="round,pad=0.06",
                facecolor=fc, edgecolor='white', linewidth=2.5, zorder=2
            ))
            fs = 10.5 if len(hand) == 4 else 12.5
            ax.text(x + 0.5, y + 0.5, hand,
                    ha='center', va='center',
                    fontsize=fs, fontweight='bold', color=tc, zorder=3)

    for k, r in enumerate(RANKS):
        ax.text(k + 0.5, N + 0.1, r,
                ha='center', va='bottom', fontsize=14, fontweight='bold', color='#2c3e50')
        ax.text(-0.1, N - 1 - k + 0.5, r,
                ha='right', va='center', fontsize=14, fontweight='bold', color='#2c3e50')

    for k in range(N):
        ax.plot([k, k + 1], [N - k, N - k],     color='#95a5a6', lw=1.2, zorder=4)
        ax.plot([k, k],     [N - k, N - k - 1], color='#95a5a6', lw=1.2, zorder=4)

    ax.text(N * 0.80, N - 0.22, 'suited ▲',
            fontsize=11, color='#636e72', ha='center', style='italic', zorder=5)
    ax.text(N * 0.20, -0.22, '▽ offsuit',
            fontsize=11, color='#636e72', ha='center', style='italic', zorder=5)

    fig.text(0.5, 0.955, title,
             ha='center', va='top', fontsize=18, fontweight='bold', color='#2c3e50')
    fig.text(0.5, 0.935,
             f'Value 3-bet {c_vbet} ({c_vbet/1326*100:.1f}%)  ·  '
             f'Bluff 3-bet {c_bluff} ({c_bluff/1326*100:.1f}%)  ·  '
             f'Call {c_call} ({c_call/1326*100:.1f}%)  ·  '
             f'Fold {c_fold} ({c_fold/1326*100:.1f}%)',
             ha='center', va='top', fontsize=10.5, color='#636e72')

    legend_items = [
        mpatches.Patch(facecolor='#922b21', edgecolor='white',
                       label=f'Value 3-bet  —  {c_vbet} combos  ({c_vbet/1326*100:.1f} %)'),
        mpatches.Patch(facecolor='#e67e22', edgecolor='white',
                       label=f'Bluff 3-bet  —  {c_bluff} combos  ({c_bluff/1326*100:.1f} %)'),
        mpatches.Patch(facecolor='#1e8449', edgecolor='white',
                       label=f'Call  —  {c_call} combos  ({c_call/1326*100:.1f} %)'),
        mpatches.Patch(facecolor='#dce1e7', edgecolor='#b2bec3', label=f'Fold  —  {c_fold} combos  ({c_fold/1326*100:.1f} %)'),
    ]
    fig.legend(handles=legend_items, loc='lower center',
               bbox_to_anchor=(0.5, 0.01), ncol=4,
               fontsize=10.5, framealpha=0.97, edgecolor='#b2bec3',
               fancybox=True, handlelength=1.6, handleheight=1.2)

    fname = filename or title.replace(' ', '_').lower() + '.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

print("draw_defense_matrix defined ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BTN vs LJ open (~20%) — tight range, we are selective
# ══════════════════════════════════════════════════════════════════════════════

BTN_vs_LJ_VBET = {
    # Premium pairs
    'QQ', 'KK', 'AA',
    # Strong suited aces
    'AQs', 'AKs',
    # Offsuit
    'AKo',
}

BTN_vs_LJ_BLUFF = {
    # Nut flush blockers + clean fold to a 4-bet
    'A2s', 'A3s', 'A4s', 'A5s',
    # Offsuit king blocker
    'KQo',
}

BTN_vs_LJ_CALL = {
    # Medium pairs — good odds IP
    'JJ', 'TT', '99', '88', '77', '66', '55', '44', '33', '22',
    # Medium suited aces
    'AJs', 'ATs', 'A9s', 'A8s',
    # Offsuit broadways
    'AQo', 'AJo',
    # Suited kings
    'KQs', 'KJs', 'KTs', 'K9s',
    # Suited queens
    'QJs', 'QTs', 'Q9s',
    # Suited connectors
    'JTs', 'J9s', 'T9s', '98s', '87s', '76s', '65s', '54s',
}

draw_defense_matrix(
    BTN_vs_LJ_VBET, BTN_vs_LJ_BLUFF, BTN_vs_LJ_CALL,
    'BTN vs LJ Open — IP Defense', 'btn_vs_lj.png'
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BTN vs HJ open (~26%) — wider range, we 3-bet more
# ══════════════════════════════════════════════════════════════════════════════

BTN_vs_HJ_VBET = {
    # Premium pairs + JJ (HJ opens wider so JJ is ahead)
    'JJ', 'QQ', 'KK', 'AA',
    # Suited aces
    'AJs', 'AQs', 'AKs',
    # Offsuit
    'AKo', 'AQo',
}

BTN_vs_HJ_BLUFF = {
    # Nut flush blockers
    'A2s', 'A3s', 'A4s', 'A5s',
    # King blockers offsuit + suited
    'KQo', 'K4s', 'K3s',
}

BTN_vs_HJ_CALL = {
    # Medium pairs
    'TT', '99', '88', '77', '66', '55', '44', '33', '22',
    # Suited aces
    'ATs', 'A9s', 'A8s', 'A7s',
    # Offsuit
    'AJo', 'ATo',
    # Suited kings
    'KQs', 'KJs', 'KTs', 'K9s', 'K8s',
    # Suited queens
    'QJs', 'QTs', 'Q9s', 'Q8s',
    # Suited jacks
    'JTs', 'J9s', 'J8s',
    # Suited connectors
    'T9s', 'T8s', '98s', '87s', '76s', '65s', '54s', '43s',
}

draw_defense_matrix(
    BTN_vs_HJ_VBET, BTN_vs_HJ_BLUFF, BTN_vs_HJ_CALL,
    'BTN vs HJ Open — IP Defense', 'btn_vs_hj.png'
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BTN vs CO open (~31%) — wide range, even wider 3-bet
# ══════════════════════════════════════════════════════════════════════════════

BTN_vs_CO_VBET = {
    # Pairs TT+ (CO opens wide, TT is ahead of its range)
    'TT', 'JJ', 'QQ', 'KK', 'AA',
    # Suited aces + KQs
    'AJs', 'AQs', 'AKs', 'KQs',
    # Offsuit
    'AKo', 'AQo', 'AJo',
}

BTN_vs_CO_BLUFF = {
    # Nut flush blockers + suited connectors (draw + blocker)
    'A2s', 'A3s', 'A4s', 'A5s',
    # King blockers
    'KQo', 'K3s', 'K2s',
    # Suited connectors — equity + pressure (CO opens wide so more FE)
    '76s', '65s',
}

BTN_vs_CO_CALL = {
    # Medium pairs
    '99', '88', '77', '66', '55', '44', '33', '22',
    # Suited aces
    'ATs', 'A9s', 'A8s', 'A7s', 'A6s',
    # Offsuit
    'ATo',
    # Suited kings
    'KJs', 'KTs', 'K9s', 'K8s', 'K7s',
    # Suited queens
    'QJs', 'QTs', 'Q9s', 'Q8s',
    # Suited jacks
    'JTs', 'J9s', 'J8s', 'J7s',
    # Suited connectors / gappers
    'T9s', 'T8s', 'T7s', '98s', '97s', '87s', '86s', '75s', '64s', '54s', '53s', '43s',
}

draw_defense_matrix(
    BTN_vs_CO_VBET, BTN_vs_CO_BLUFF, BTN_vs_CO_CALL,
    'BTN vs CO Open — IP Defense', 'btn_vs_co.png'
)

---
## SB — OOP Defense (facing an open)

The SB is **Out of Position** against all openers → more conservative strategy than the BTN.
- Tighter, more polarized 3-bet range (we don't want to call OOP with marginal hands)
- Narrower call range (paying to play OOP is costly)
- The later the opener's position (wide range), the wider we 3-bet to compensate

> The big difference with the BTN: we fold many more medium hands than the BTN would.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SB vs LJ open (~20%) — tight LJ range → very selective 3-bet
# ══════════════════════════════════════════════════════════════════════════════

SB_vs_LJ_VBET = {
    # Pairs QQ+ (LJ tight range ~20%, QQ is clearly ahead)
    'QQ', 'KK', 'AA',
    # Premium suited aces
    'AQs', 'AKs',
    # Offsuit
    'AKo',
}

SB_vs_LJ_BLUFF = {
    # Nut flush blockers — clean fold to a 4-bet without sacrificing equity
    'A2s', 'A3s', 'A4s', 'A5s',
    # Offsuit king blocker
    'KQo',
}

SB_vs_LJ_CALL = {
    # Pairs: JJ (too strong to fold, too risky to 3-bet vs a tight range)
    'JJ', 'TT', '99',
    # Suited aces
    'AJs', 'ATs',
    # Suited kings / queens / jacks
    'KQs', 'KJs', 'QJs', 'JTs',
    # Suited connectors (keep playability while OOP)
    'T9s', '98s', '87s',
}

draw_defense_matrix(
    SB_vs_LJ_VBET, SB_vs_LJ_BLUFF, SB_vs_LJ_CALL,
    'SB vs LJ Open — OOP Defense', 'sb_vs_lj.png'
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SB vs HJ open (~26%) — intermediate range
# ══════════════════════════════════════════════════════════════════════════════

SB_vs_HJ_VBET = {
    # Pairs QQ+ (HJ range ~26%, QQ+ clearly ahead)
    'QQ', 'KK', 'AA',
    # JJ: slightly +EV 3-bet vs HJ range
    'JJ',
    # Strong suited aces
    'AJs', 'AQs', 'AKs',
    # Offsuit
    'AKo', 'AQo',
}

SB_vs_HJ_BLUFF = {
    # Nut flush blockers — clean fold to a 4-bet
    'A2s', 'A3s', 'A4s', 'A5s',
    # Offsuit king blocker
    'KQo', 'K4s',
}

SB_vs_HJ_CALL = {
    # Medium pairs
    'TT', '99', '88',
    # Suited aces
    'ATs',
    # Offsuit broadways
    'AJo',
    # Suited kings / queens / jacks
    'KQs', 'KJs', 'QJs', 'JTs',
    # Suited connectors
    'T9s',
}

draw_defense_matrix(
    SB_vs_HJ_VBET, SB_vs_HJ_BLUFF, SB_vs_HJ_CALL,
    'SB vs HJ Open — OOP Defense', 'sb_vs_hj.png'
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SB vs CO open (~31%) — wide range, more aggressive 3-bet than vs HJ
# ══════════════════════════════════════════════════════════════════════════════

SB_vs_CO_VBET = {
    # Pairs JJ+ (CO opens ~31%, JJ is ahead)
    'JJ', 'QQ', 'KK', 'AA',
    # Suited aces + KQs
    'AJs', 'AQs', 'AKs', 'KQs',
    # Offsuit
    'AKo', 'AQo', 'AJo',
}

SB_vs_CO_BLUFF = {
    # Nut flush blockers
    'A2s', 'A3s', 'A4s', 'A5s',
    # King blockers suited + offsuit
    'K3s', 'K4s', 'KQo',
    # Suited semi-bluff (increased FE vs CO's wide range)
    '76s',
}

SB_vs_CO_CALL = {
    # Medium pairs
    'TT', '99', '88', '77',
    # Suited aces
    'ATs', 'A9s',
    # Offsuit
    'AQo', 'AJo',
    # Suited kings / queens / jacks
    'KJs', 'KTs', 'QJs', 'JTs',
    # Suited connectors
    'T9s', '98s',
}

# Value/call deduplication
for h in SB_vs_CO_VBET:
    SB_vs_CO_CALL.discard(h)

draw_defense_matrix(
    SB_vs_CO_VBET, SB_vs_CO_BLUFF, SB_vs_CO_CALL,
    'SB vs CO Open — OOP Defense', 'sb_vs_co.png'
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SB vs BTN open (~48%) — BTN opens very wide, we 3-bet wide OOP
# ══════════════════════════════════════════════════════════════════════════════

SB_vs_BTN_VBET = {
    # Pairs TT+ (BTN wide range → TT ahead)
    'TT', 'JJ', 'QQ', 'KK', 'AA',
    # Strong suited aces + KQs
    'AJs', 'AQs', 'AKs', 'KQs',
    # Offsuit broadways
    'AKo', 'AQo',
}

SB_vs_BTN_BLUFF = {
    # Nut flush blockers — clean fold to a 4-bet
    'A2s', 'A3s', 'A4s', 'A5s',
    # King blockers
    'K2s', 'K3s', 'KQo',
    # Suited connectors semi-bluff (BTN opens very wide → high FE)
    '76s', '65s',
}

SB_vs_BTN_CALL = {
    # Medium pairs (call OOP: keep the pot small, realize equity)
    '99', '88', '77', '66',
    # Suited aces
    'ATs', 'A9s',
    # Offsuit broadways
    'AJo', 'AQo',
    # Suited kings
    'KJs', 'KTs',
    # Suited queens / jacks
    'QJs', 'QTs', 'JTs',
    # Suited connectors
    'T9s', '98s', '87s',
}

# Note: AQo appears in both call AND value — we keep it only in vbet
SB_vs_BTN_CALL.discard('AQo')

draw_defense_matrix(
    SB_vs_BTN_VBET, SB_vs_BTN_BLUFF, SB_vs_BTN_CALL,
    'SB vs BTN Open — OOP Defense', 'sb_vs_btn.png'
)

---
## BB (Big Blind) — Defense (facing an open)

The BB has already invested 1BB → **better pot odds** than the SB → defends much wider.  
Range adjusted to the opener's position (the wider the opener, the wider the BB calls/3-bets).

- 🔴 **Value 3-bet**: very strong hands, ahead of the opener's range
- 🟠 **Bluff 3-bet**: nut flush blockers (low suited Aces) + offsuit king blockers
- 🟢 **Call**: very wide — pairs, suited hands, connectors, reasonable offsuit hands
- ⚪ **Fold**: only hands that are truly too weak to defend

> BB golden rule: never fold too easily (pot odds require a wide defense).  
> Against the BTN in particular, the BB calls down to very marginal hands.

> ⚠️ **BB vs SB exception**: the BB is **in position (IP)** post-flop against the SB (SB acts first).  
> → BB defends even wider than against the BTN: maximum call range, fold only absolute trash (~15%).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BB vs LJ open (~20%) — tight LJ range → conservative BB 3-bet, wide call
# ══════════════════════════════════════════════════════════════════════════════

BB_vs_LJ_VBET = {
    # Pairs QQ+ (LJ tight range, QQ clearly ahead)
    'QQ', 'KK', 'AA',
    # Premium suited aces
    'AQs', 'AKs',
    # Offsuit
    'AKo',
}

BB_vs_LJ_BLUFF = {
    # Nut flush blockers — clean fold to a 4-bet
    'A2s', 'A3s', 'A4s', 'A5s',
    # Offsuit king blocker
    'K5s', 'KQo',
}

BB_vs_LJ_CALL = {
    # Pairs — good equity IP vs tight LJ range
    'JJ', 'TT', '99', '88', '77', '66', '55', '44', '33', '22',
    # Suited aces
    'AJs', 'ATs', 'A9s', 'A8s', 'A7s', 'A6s',
    # Offsuit aces
    'AQo', 'AJo', 'ATo',
    # Suited kings
    'KQs', 'KJs', 'KTs', 'K9s', 'K8s',
    # Offsuit kings
    'KQo', 'KJo',
    # Suited queens / jacks
    'QJs', 'QTs', 'Q9s',
    'JTs', 'J9s',
    # Offsuit
    'QJo',
    # Suited connectors
    'T9s', 'T8s', '98s', '97s', '87s', '86s', '76s', '75s', '65s', '64s', '54s', '53s', '43s',
}

draw_defense_matrix(
    BB_vs_LJ_VBET, BB_vs_LJ_BLUFF, BB_vs_LJ_CALL,
    'BB vs LJ Open — OOP Defense', 'bb_vs_lj.png'
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BB vs HJ open (~26%) — wide call, selective 3-bet
# ══════════════════════════════════════════════════════════════════════════════

BB_vs_HJ_VBET = {
    # Pairs QQ+ (+ JJ slightly +EV vs HJ)
    'JJ', 'QQ', 'KK', 'AA',
    # Suited aces
    'AQs', 'AKs',
    # Offsuit
    'AKo', 'AQo',
}

BB_vs_HJ_BLUFF = {
    # Nut flush blockers
    'A2s', 'A3s', 'A4s', 'A5s',
    # King blockers
    'K4s', 'K5s', 'KQo',
}

BB_vs_HJ_CALL = {
    # Pairs
    'TT', '99', '88', '77', '66', '55', '44', '33', '22',
    # Suited aces
    'AJs', 'ATs', 'A9s', 'A8s', 'A7s', 'A6s',
    # Offsuit aces
    'AJo', 'ATo', 'A9o',
    # Suited kings
    'KQs', 'KJs', 'KTs', 'K9s', 'K8s', 'K7s', 'K6s',
    # Offsuit kings
    'KJo', 'KTo',
    # Suited queens
    'QJs', 'QTs', 'Q9s', 'Q8s',
    # Offsuit queens
    'QJo', 'QTo',
    # Suited jacks
    'JTs', 'J9s', 'J8s',
    # Offsuit jacks
    'JTo',
    # Suited tens
    'T9s', 'T8s',
    # Suited connectors
    '98s', '97s', '87s', '86s', '76s', '75s', '65s', '64s', '54s', '53s', '43s',
}

draw_defense_matrix(
    BB_vs_HJ_VBET, BB_vs_HJ_BLUFF, BB_vs_HJ_CALL,
    'BB vs HJ Open — OOP Defense', 'bb_vs_hj.png'
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BB vs CO open (~31%) — still a very wide call, well-constructed 3-bet
# ══════════════════════════════════════════════════════════════════════════════

BB_vs_CO_VBET = {
    # Pairs JJ+
    'JJ', 'QQ', 'KK', 'AA',
    # Suited aces
    'AJs', 'AQs', 'AKs',
    # Offsuit
    'AKo', 'AQo', 'AJo',
}

BB_vs_CO_BLUFF = {
    # Nut flush blockers
    'A2s', 'A3s', 'A4s', 'A5s',
    # King blockers
    'K3s', 'K4s', 'KQo',
    # Semi-bluff (FE vs wide CO range)
    '76s', '65s',
}

BB_vs_CO_CALL = {
    # Pairs
    'TT', '99', '88', '77', '66', '55', '44', '33', '22',
    # Suited aces
    'ATs', 'A9s', 'A8s', 'A7s', 'A6s',
    # Offsuit aces
    'ATo', 'A9o', 'A8o',
    # Suited kings
    'KQs', 'KJs', 'KTs', 'K9s', 'K8s', 'K7s', 'K6s',
    # Offsuit kings
    'KJo', 'KTo', 'K9o',
    # Suited queens
    'QJs', 'QTs', 'Q9s', 'Q8s',
    # Offsuit queens
    'QJo', 'QTo',
    # Suited jacks
    'JTs', 'J9s', 'J8s', 'J7s',
    # Offsuit jacks
    'JTo', 'J9o',
    # Suited tens
    'T9s', 'T8s', 'T7s',
    # Offsuit
    'T9o',
    # Suited connectors / gappers
    '98s', '97s', '96s', '87s', '86s', '85s',
    '75s', '74s', '64s', '63s', '53s', '54s', '43s', '32s',
}

for h in BB_vs_CO_VBET:
    BB_vs_CO_CALL.discard(h)

draw_defense_matrix(
    BB_vs_CO_VBET, BB_vs_CO_BLUFF, BB_vs_CO_CALL,
    'BB vs CO Open — OOP Defense', 'bb_vs_co.png'
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BB vs BTN open (~48%) — BTN steals very wide, BB must defend massively
# ══════════════════════════════════════════════════════════════════════════════

BB_vs_BTN_VBET = {
    # Premium pairs
    'TT', 'JJ', 'QQ', 'KK', 'AA',
    # Suited aces + KQs
    'AQs', 'AKs', 'KQs',
    # Offsuit
    'AKo', 'AQo',
}

BB_vs_BTN_BLUFF = {
    # Nut flush blockers
    'A2s', 'A3s', 'A4s', 'A5s',
    # Suited king blockers
    'K2s', 'K3s', 'K4s',
    # Offsuit blocker
    'KQo',
    # Suited semi-bluff (very wide BTN range → max FE)
    '76s', '65s', '54s',
}

BB_vs_BTN_CALL = {
    # Pairs
    '99', '88', '77', '66', '55', '44', '33', '22',
    # Suited aces
    'AJs', 'ATs', 'A9s', 'A8s', 'A7s', 'A6s',
    # Offsuit aces
    'AJo', 'ATo', 'A9o', 'A8o', 'A7o',
    # Suited kings
    'KQs', 'KJs', 'KTs', 'K9s', 'K8s', 'K7s', 'K6s', 'K5s',
    # Offsuit kings
    'KJo', 'KTo', 'K9o', 'K8o',
    # Suited queens
    'QJs', 'QTs', 'Q9s', 'Q8s', 'Q7s',
    # Offsuit queens
    'QJo', 'QTo', 'Q9o',
    # Suited jacks
    'JTs', 'J9s', 'J8s', 'J7s', 'J6s',
    # Offsuit jacks
    'JTo', 'J9o',
    # Suited tens
    'T9s', 'T8s', 'T7s', 'T6s',
    # Offsuit
    'T9o',
    # Low suited connectors / gappers
    '98s', '97s', '96s', '87s', '86s', '85s',
    '75s', '74s', '64s', '63s', '53s', '52s', '43s', '42s', '32s',
}

# Dedup with value
for h in BB_vs_BTN_VBET:
    BB_vs_BTN_CALL.discard(h)

draw_defense_matrix(
    BB_vs_BTN_VBET, BB_vs_BTN_BLUFF, BB_vs_BTN_CALL,
    'BB vs BTN Open — OOP Defense', 'bb_vs_btn.png'
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BB vs SB raise (~18% polarized range) — BB is IP! Defends very wide.
# ══════════════════════════════════════════════════════════════════════════════
# SB polarized raise range: value (88+, ATs+, AJo+, KQs, KJo+) + bluffs (offsuit Axo, Kxo)
# BB IP + pot odds → maximum call range, fold only absolute trash (~15%)

BB_vs_SB_VBET = {
    # Hands that want a big pot vs SB's polarized range
    'QQ', 'KK', 'AA',
    'AQs', 'AKs',
    'AKo', 'AQo',  # dominates SB bluffs (Axo) and SB value
}

BB_vs_SB_BLUFF = {
    # Nut flush blockers — clean fold if SB 4-bets
    'A2s', 'A3s', 'A4s', 'A5s',
}

BB_vs_SB_CALL = {
    # Pairs (IP → call JJ, no need to 3-bet for protection)
    'JJ', 'TT', '99', '88', '77', '66', '55', '44', '33', '22',
    # Suited aces
    'AJs', 'ATs', 'A9s', 'A8s', 'A7s', 'A6s',
    # Offsuit aces — wide call because IP + equity vs SB bluffs
    'AJo', 'ATo', 'A9o', 'A8o', 'A7o', 'A6o', 'A5o', 'A4o', 'A3o', 'A2o',
    # Suited kings (almost all K2s+)
    'KQs', 'KJs', 'KTs', 'K9s', 'K8s', 'K7s', 'K6s', 'K5s', 'K4s', 'K3s', 'K2s',
    # Offsuit kings
    'KQo', 'KJo', 'KTo', 'K9o', 'K8o', 'K7o',
    # Suited queens
    'QJs', 'QTs', 'Q9s', 'Q8s', 'Q7s', 'Q6s', 'Q5s', 'Q4s',
    # Offsuit queens
    'QJo', 'QTo', 'Q9o', 'Q8o',
    # Suited jacks
    'JTs', 'J9s', 'J8s', 'J7s', 'J6s', 'J5s',
    # Offsuit jacks
    'JTo', 'J9o', 'J8o',
    # Suited tens
    'T9s', 'T8s', 'T7s', 'T6s', 'T5s',
    # Offsuit tens
    'T9o', 'T8o',
    # Suited connectors / gappers (everything with IP playability)
    '98s', '97s', '96s', '95s',
    '87s', '86s', '85s', '84s',
    '76s', '75s', '74s',
    '65s', '64s', '63s',
    '54s', '53s', '52s',
    '43s', '42s', '32s',
    # Offsuit connectors
    '98o', '87o',
}

draw_defense_matrix(
    BB_vs_SB_VBET, BB_vs_SB_BLUFF, BB_vs_SB_CALL,
    'BB vs SB Raise — IP Defense (BB in position!)', 'bb_vs_sb.png'
)